# DBR 14.2 Getting Started: Spatial SQL Preview [v1]

> Get up and running with the new `ST_` functions. This will help you validate your environment, discuss supported data formats and types, highlight some of the functions, and then provide an intro to h3 indexing to assist with spatial joins, e.g. point-in-polygon using `st_contains`.

__Notes:__

<p/>

1. This is focused on v1 of the preview which is SQL API only
1. Requires Photon DBR 14.2
1. Serverless / DBSQL will not be available until after the 2023 Holiday Release Restricted Period
1. Assumes you have already signed the terms of services for the preview

---
__Author:__ Michael Johns | <mjohns@databricks.com>  
__Last Modified:__ 09 NOV 2023

In [0]:
displayHTML("""
<span style="color:red;font-weight:200;font-size:20px">
    Please ensure an authorized team member has accepted the Private Preview terms of service [by replying to a Databricks initiated email].
</span>
<span style="font-size:18px;line-height:150%">
  <ul>
<li>The product is in private preview, is not intended for use in production, and is provided AS-IS consistent with your agreement with Databricks.</li>
<li>Although the preview is not intended for use in production, you may still incur charges for platform usage DBUs.</li>
<li>Non-public information about the preview (including the fact that there is a preview for the feature/product itself) is confidential.</li>
<li>We may change or discontinue the preview at any time without notice. We may also choose not to make the preview generally commercially available.</li>
<li>We may charge for this preview offering in the future.</li>
<li>Previews are not included in the SLA and do not have formal support. If you have questions or feedback, please reach out to your Databricks contact.</li>
  </ul>
</span>
""")

## [1] Setup Environment

> Ensure you have the preview flag enabled for DBR 14.2 and that you can view available functions.

__Optional: Install KeplerGL for inline rendering__

> Run the following hidden cell to setup function `map_render(df:DataFrame, geom_col:str)`. _NOTICE: this code block includes `%pip` and restarts python kernel, so it is run at the top of notebook. Also, you may not have appropriate permissions to install external libraries (based on cluster policies or cluster type); if so, you can comment out or otherwise skip execution of the rendering cells._

In [0]:
# - optionally comment out to not use kepler
%pip install keplergl==0.3.2 --quiet
dbutils.library.restartPython() # <- restart python kernel

In [0]:
use_kepler = True # <- optionally set to False to not use kepler

_Here are optional helper functions for map rendering [in hidden code block]._

> These will essentially skip if (1) you did not install kepler or (2) you set `use_kepler=False`.

In [0]:
try:
  print(f"use keperl? {use_kepler}")

  if not use_kepler:
    print("customer disabled helpers...")
    raise Exception("customer disabled helpers...")

  # -- setup helper functions
  from dataclasses import dataclass
  from dataclasses import field
  from enum import Enum
  from keplergl import KeplerGl
  import math
  from pyspark.sql import DataFrame
  import re
  from typing import List

  kepler_height=800
  kepler_width=1200


  def display_kepler(kmap:KeplerGl, height:int=kepler_height, width:int=kepler_width) -> None:
    """
    Convenience function to render map in kepler.gl
    """
    decoded = (
        kmap._repr_html_()
        .decode("utf-8")
        .replace(".height||400", f".height||{height}")
        .replace(".width||400", f".width||{width}")
    )
    ga_script_redacted = re.sub(
        r"\<script\>\(function\(i,s,o,g,r,a,m\).*?GoogleAnalyticsObject.*?(\<\/script\>)",
        "",
        decoded,
        flags=re.DOTALL,
    )
    async_script_redacted = re.sub(
        r"s\.a\.createElement\(\"script\",\{async.*?\}\),",
        "",
        ga_script_redacted,
        flags=re.DOTALL,
    )
    displayHTML(async_script_redacted)


  class RENDER_TYPE(Enum):
    """Specify expected type of a 'render_col' in DFMapItem."""
    GEOMETRY = 100
    H3_INT = 200
    H3_STR = 300


  class GEO_FORMAT(Enum):
    """Specify expected type of a 'render_col' in DFMapItem [when RENDER_TYPE = GEOMETRY]."""
    WKT = 1
    WKB = 2
    GEOJSON = 3
    EWKB = 4

  @dataclass
  class DFMapItem:
    """Class for holding some properties for rendering a map."""
    df:DataFrame
    render_col:str
    render_type:RENDER_TYPE
    geo_format:GEO_FORMAT=None
    layer_name:str=None
    zoom_calc_sample_limit:int=None
    exclude_cols:list=field(default_factory=list)


  @dataclass
  class ZoomInfo:
    map_x:float
    map_y:float
    map_zoom:float

  default_ZoomInfo = ZoomInfo(0.0, 0.0, 3.0)


  def calc_ZoomInfo(dfMapItem:DFMapItem, debug_level:int=0) -> ZoomInfo:
    """
    Example output of debug_level=1
    {'xmin': -100.5, 'ymin': 50.05, 'xmax': -100.25, 'ymax': 50.5, 'centroid_x': -100.375, 'centroid_y': 50.275, 'pnt_sw': 'POINT(-100.5 50.05)', 'pnt_nw': 'POINT(-100.5 50.5)', 'pnt_se': 'POINT(-100.25 50.05)', 'pnt_ne': 'POINT(-100.25 50.5)',  'width_meters': 17905.33401827115, 'height_meters': 50055.461462782696, 'max_meters': 50055.461462782696, 'zoom': 9.5} 
    """
    # - handle zoom sample
    df_samp = dfMapItem.df
    samp_limit = dfMapItem.zoom_calc_sample_limit
    if samp_limit is not None:
      cnt = df_samp.count()
      if samp_limit < cnt:
        df_samp = (
          df_samp
            .dropna(dfMapItem.render_col)
            .sample(float(samp_limit)/float(cnt))
            .limit(samp_limit)
        )

    # - handle h3
    geom_col = dfMapItem.render_col
    if dfMapItem.render_type in [RENDER_TYPE.H3_INT,RENDER_TYPE.H3_STR]:
      geom_col = "h3_geom"
      df_samp = df_samp.withColumn(geom_col, F.expr(f"h3_boundaryaswkb({dfMapItem.render_col})"))

    # standardize to SRID=4326
    if dfMapItem.geo_format is not None:
      from_str = None
      if dfMapItem.geo_format == GEO_FORMAT.WKT:
        from_str='wkt'
      elif dfMapItem.geo_format == GEO_FORMAT.WKB:
        from_str='wkb'
      elif dfMapItem.geo_format == GEO_FORMAT.GEOJSON:
        from_str='geojson'
      elif dfMapItem.geo_format == GEO_FORMAT.EWKB:
        from_str='ewkb'
      # ... only do the operation if from_clause identified
      if from_str is not None:
        srid = df_samp.select(F.expr(f"st_srid(st_geomfrom{from_str}({geom_col}))")).first()[0]
        if srid is not None and srid > 0 and srid != 4326:
          df_samp = (
            df_samp
              .selectExpr(
                f"st_asbinary(st_transform(st_geomfrom{from_str}({geom_col}, 4326))) as {geom_col}", 
                f"* except({geom_col})"
              )
          )

    d = (
      df_samp
        # - xy min/max
        .select( 
          F.expr(f"st_xmin({geom_col}) as xmin"), 
          F.expr(f"st_ymin({geom_col}) as ymin"),
          F.expr(f"st_xmax({geom_col}) as xmax"),
          F.expr(f"st_ymax({geom_col}) as ymax")
        )
      .groupBy()
        .agg(
          F.min("xmin").alias("xmin"),
          F.min("ymin").alias("ymin"),
          F.max("xmax").alias("xmax"),
          F.max("ymax").alias("ymax")
        )
        # - centroid xy ranges
        .withColumn("centroid_x", F.expr("(xmin + xmax) / 2.0"))
        .withColumn("centroid_y", F.expr("(ymin + ymax) / 2.0"))  
        .withColumn("pnt_sw", F.expr("st_astext(st_point(xmin,ymin))"))
        .withColumn("pnt_nw", F.expr("st_astext(st_point(xmin,ymax))"))
        .withColumn("pnt_se", F.expr("st_astext(st_point(xmax,ymin))"))
        .withColumn("pnt_ne", F.expr("st_astext(st_point(xmax,ymax))"))
        .withColumn(
          "width_meters", 
          F.expr("st_geoglength(st_astext(st_makeline(array( st_geomfromtext(pnt_sw), st_geomfromtext(pnt_se) ))))")
        )
        .withColumn(
          "height_meters", 
          F.expr("st_geoglength(st_astext(st_makeline(array( st_geomfromtext(pnt_sw), st_geomfromtext(pnt_nw) ))))")
        )
        .withColumn(
          "max_meters", 
          F
            .when(F.expr("width_meters >= height_meters"), F.col("width_meters"))
            .otherwise(F.col("height_meters"))
        )
        # - zoom
        # https://wiki.openstreetmap.org/wiki/Slippy_map_tilenames#Resolution_and_Scale
        # 1cm = ~.4in
        # assume 16cm = ~6in (height of viewport)
        # but mapbox tiles are 512px instead of 256px, so divide by 2 [h=8 tiles]
        .withColumn(
          "zoom",
          F
            .when(F.expr("max_meters < 21.2 * 8"), F.lit(18))
            .when(F.expr("max_meters < 42.3 * 8"), F.lit(17))
            .when(F.expr("max_meters < 84.6 * 8"), F.lit(16))
            .when(F.expr("max_meters < 169 * 8"), F.lit(15))
            .when(F.expr("max_meters < 339 * 8"), F.lit(14))
            .when(F.expr("max_meters < 677 * 8"), F.lit(13))
            .when(F.expr("max_meters < 1.35 * 1000 * 8"), F.lit(12))
            .when(F.expr("max_meters < 2.7  * 1000 * 8"), F.lit(11))
            .when(F.expr("max_meters < 5.4  * 1000 * 8"), F.lit(10))
            .when(F.expr("max_meters < 10.8 * 1000 * 8"), F.lit(9))
            .when(F.expr("max_meters < 21.7 * 1000 * 8"), F.lit(8))
            .when(F.expr("max_meters < 43.3 * 1000 * 8"), F.lit(7))
            .when(F.expr("max_meters < 86.7 * 1000 * 8"), F.lit(6))
            .when(F.expr("max_meters < 173  * 1000 * 8"), F.lit(5))
            .when(F.expr("max_meters < 347  * 1000 * 8"), F.lit(4))
            .when(F.expr("max_meters < 693  * 1000 * 8"), F.lit(3))
            .when(F.expr("max_meters < 1387 * 1000 * 8"), F.lit(2))
            .when(F.expr("max_meters < 2773 * 1000 * 8"), F.lit(1))
            .otherwise(F.lit(0))
        )
    ).first().asDict()

    (debug_level > 0) and print(d,"\n")
    return ZoomInfo(d['centroid_x'], d['centroid_y'], d['zoom'])


  def map_render_dfMapItems(*dfMapItems:List[DFMapItem], 
               override_ZoomInfo:ZoomInfo=None,
               kepler_map_style:str='dark',
               debug_level:int=0) ->None:
    """
    Calls `display_kepler` using conventions.
    - Calculates center lat/lon and zoom level [based on first layer passed], 
      if override ZoomInfo not specified
    - Renders one or more passed Spark DFMapItems,
      each will be a separate layer
    - Must specify render col and RENDER_TYPE
    - Can use specified layer name in DFMapItem;
      otherwise, will be generated
    - Can specify a render sample limit in DFMapItem
    - Can specify a zoom calc sample limit in DFMapItem;
      otherwise it will be all
    """

    layers = {}
    zoomInfo = default_ZoomInfo

    for layer_num, dfMapItem in enumerate(dfMapItems):
      # - zoom info [first layer]
      if layer_num == 0:
       zoomInfo = calc_ZoomInfo(dfMapItem, debug_level=debug_level)

      # - layer name
      layer_name = dfMapItem.layer_name
      if layer_name is None:
        layer_name = f"layer_{layer_num}"

      # - data
      if dfMapItem.render_type in [RENDER_TYPE.GEOMETRY]:
        # handle binary serialization
        geo_format = dfMapItem.geo_format
        if geo_format is not None and geo_format in [GEO_FORMAT.WKB, GEO_FORMAT.EWKB]:
          layers[layer_name] = (
            dfMapItem
              .df
                .drop(*dfMapItem.exclude_cols)
              .toPandas()
                .to_csv(None, index=False)
          )
        else:
          layers[layer_name] = (
            dfMapItem
              .df
                .drop(*dfMapItem.exclude_cols)
              .toPandas()
          )
      elif dfMapItem.render_type in [RENDER_TYPE.H3_STR]:
          layers[layer_name] = (
            dfMapItem
              .df
                .drop(*dfMapItem.exclude_cols)
              .toPandas()
          )
      elif dfMapItem.render_type == RENDER_TYPE.H3_INT:
        layers[layer_name] = (
          dfMapItem
            .df
              .selectExpr(
                f"h3_h3tostring({dfMapItem.render_col}) as {dfMapItem.render_col}", 
                f"* except({dfMapItem.render_col})"
              )
              .drop(*dfMapItem.exclude_cols)
            .toPandas()
        )
      
    return display_kepler(
      KeplerGl(
        config={ 
          'version': 'v1', 
          'mapState': {
            'longitude': zoomInfo.map_x, 
            'latitude': zoomInfo.map_y, 
            'zoom': zoomInfo.map_zoom
          }, 
          'mapStyle': {'styleType': kepler_map_style},
          'options': {'readOnly': False, 'centerMap': True}
        },
        data=layers,
        show_docs=False,
      )
    )


  def map_render(df:DataFrame, geom_col:str, geo_format:GEO_FORMAT=None, exclude_cols:list=[], override_ZoomInfo:ZoomInfo=None, kepler_map_style='dark', debug_level:int=0)   ->None:
    """
    Render a Spark Dataframe, using geometry col for center and zoom,
    if overrides not specified. 
    """
    map_render_dfMapItems(DFMapItem(df, geom_col, RENDER_TYPE.GEOMETRY, geo_format=geo_format, exclude_cols=exclude_cols), 
               override_ZoomInfo=override_ZoomInfo, 
               kepler_map_style=kepler_map_style,
               debug_level=debug_level)

  print("---")
  print("def map_render(df:DataFrame, geom_col:str)")
  print("def map_render_dfMapItems(*dfMapItems:List[DFMapItem])")

except Exception:
  print("... `map_render` functions not available.")
  pass

In [0]:
## - enable the spatial preview functions (for this session)
spark.conf.set("spark.databricks.geo.st.enabled", True)

## - v1 supports SQL language only for the new ST_ functions;
##   however, other built-in functions are available 
from pyspark.databricks.sql import functions as dbf
from pyspark.sql import functions as F

In [0]:
%sql 
-- you can also enable spatial preview functions via sql
-- set spark.databricks.geo.st.enabled=true

_Even if you can successfully list `ST_` functions,_ __if you are not on a photon DBR, you will see an exception when trying to invoke them.__

In [0]:
%sql 
show functions like 'st_*'

__Here is some initial [simple] data...__

> These will help us quickly demonstrate the new `ST_` functions.

In [0]:
%sql 

CREATE OR REPLACE TEMPORARY VIEW samp_lat_lon AS (
  SELECT -75.5 as longitude, 40.5 as latitude
);

CREATE OR REPLACE TEMPORARY VIEW samp_pnt AS (
  SELECT st_astext(st_point(longitude, latitude)) as pnt
  FROM samp_lat_lon
);

CREATE OR REPLACE TEMPORARY VIEW samp_poly AS (
  SELECT 1 as row_id, 'POLYGON((-115.427990375581 32.5700043540444, -115.427944725767 32.5700982923276, -115.428003926578 32.5701189200886, -115.428049576338 32.5700249817816, -115.427990375581 32.5700043540444))' as g 
);

CREATE OR REPLACE TEMPORARY VIEW samp_line AS (
  SELECT 1 as row_id, 'LINESTRING(-100.25 50.5,-100.50 50.40,-100.25 50.35,-100.30 50.05)' as g
);

__Let's render the initial data__

> Below is a screenshot of the sample view data.

In [0]:
displayHTML("""<img src="..." width="100%"/>""")

In [0]:
## -- uncomment to render the views
# map_render(spark.table("samp_poly"),"g")
# map_render(spark.table("samp_line"),"g")
# map_render(spark.table("samp_pnt"), "pnt")

## [2] Supported Interchange Formats

> Most Databricks spatial expressions support reading directly from Well-known Text (WKT), Well-known Binary (WKB), Extended Well-known Binary (EWKB), and GeoJSON. You can also convert to native geospatial types (Geography and Geometry) and use these within a query. Materializing Geography and Geometry values or returning these values in a resultset is not currently supported, you need to convert to WKT, WKB, EWKB, or GeoJSON to do so. For example if you want to get the perimeter of a column with WKT aerial geometries, you can invoke `ST_Perimeter(<wkt_col>)` directly versus `ST_Perimeter(ST_GeomFromText(<wkt_col))`. __Note: you can always use other libraries with DBR to handle various file formates, e.g. Shapefiles and FileGeoDatabases, but that is not the focus of this Quickstart.__ 

## Latitude / Longitude

> `st_point` is available specifically for generating a point geometry from latitude / longitude. Also, point geometries can access lat and lon through `st_y` (latitude) and `st_x` (longitude).

In [0]:
%sql 
-- start from lat/lon data, return geometry wrapped as WKT
select st_astext(st_point(longitude, latitude)) as pnt from samp_lat_lon;

In [0]:
%sql 
-- start from WKT point data, return lat/lon
select st_x(pnt) as lon, st_y(pnt) as lat from samp_pnt;

### WKT

> Well-Known Text interchange format is supported. WKT may be POINT, LINESTRING, POLYGON as well as MULTI- variations of each as well as GEOMETRY COLLECTIONS, more [here](https://libgeos.org/specifications/wkt/).

In [0]:
%sql 
-- perimeter of aerial WKT, perimeter units are in degrees for WGS84
select st_perimeter(g) as poly_perim, * from samp_poly

### WKB / EWKB

> Well-Known Binary interchange format is supported. It is essentially the binary encoded equivalent of WKT [discussed above]. Also, functions `st_asewkb` and `st_geomfromewkb` are available for Extended Well-Known Binary, more [here](https://libgeos.org/specifications/wkb/#extended-wkb).

In [0]:
%sql 
-- start from interchange data, perimeter units are in degrees for WGS84
-- we are converting to WKB [Binary] from 'samp_poly' WKT [String] for this
with samp_wkb as (
  select * except (g), st_asbinary(st_geomfromtext(g)) as g from samp_poly
) 
select st_perimeter(g) as poly_perim, * from samp_wkb

### GeoJSON

> GeoJSON interchange format is supported, more [here](https://libgeos.org/specifications/geojson/).

In [0]:
%sql 
-- start from interchange data, perimeter units are in degrees for WGS84
-- we are converting to GeoJSON from 'samp_poly' WKT [String] for this
with samp_geojson as (
  select * except (g), st_asgeojson(st_geomfromtext(g)) as g from samp_poly
) 
select st_perimeter(g) as poly_perim, * from samp_geojson

## Geometry + Geography Types

>  In a chained query the return type standardized to the in-memory types has benefits, e.g. in the chain `ST_Buffer(ST_Envelope(<geojson_col>, <radius>))`, both `ST_Buffer` and `ST_Envelope` return a GEOMETRY type, with `ST_Buffer` only accepting GEOMETRY param and `ST_Envelope` supporting overloads, so the standardization to GEOMETRY lends to efficiency since the in-memory type can be reused once generated. __For the preview, you cannot write Geography and Geometry types or display them as these are in-memory only, so you need to convert to WKT, WKB, or GeoJSON prior to storing or including in final results.__

_Exception raised if you attempt to return or store GEOGRAPHY or GEOMETRY data types:_

```
SparkUnsupportedOperationException: [ST_UNSUPPORTED_RETURN_TYPE] The GEOGRAPHY and GEOMETRY data types
cannot be returned in queries. Use one of the following SQL expressions to convert them to standard 
interchange formats: "st_asbinary", "st_astext", or "st_asgeojson". SQLSTATE: 0A000
```


In [0]:
%sql 
-- start from interchange data, buffer units are in degrees for WGS84
-- NOTICE: chaining in-memory geometry type, but return is back to interchange format
select st_astext(st_buffer(st_envelope(g), 0.001)) as buf_g, * from samp_poly

In [0]:
%sql 
-- UNCOMMENT TO THROW THE EXCEPTION
-- due to not converting to an interchange format
-- select st_buffer(st_envelope(g), 0.001) as buf_g, * from samp_poly


> When you want to work with in-memory GEOGRAPHY types, e.g. for `ST_GeogArea`, `ST_GeogLength`, and `ST_GeogPerimeter` available in the preview, you can construct using calls like `ST_GeogFromWKT`, `ST_GeogFromWKB`, and `ST_GeogFromGeoJSON`. __`ST_Geog*` calculations are spherical (results in meters) vs cartesian in the provided units (results in degrees for 4326).__ _While cartesian calculations are faster, they are only suitable for shorter areas of calculation or comparison. And you need to consider that a degree of latitude remains fairly constant from the equator to the poles; however a degree of longitude can vary greatly as one approaches the poles and the meridians converge._

In [0]:
%sql 
-- NOTICE: difference between `st_geogperimeter` [meters] and `st_perimeter` [units: e.g. degrees]
select st_geogperimeter(g) as dbx_perim_meters, st_perimeter(g) as dbx_perim_units, * from samp_poly

In [0]:
%sql 
-- NOTICE: difference between `st_geogarea` [meters] and `st_area` [units: e.g. degrees]
select st_geogarea(g) as dbx_area_meters, st_area(g) as dbx_area_units, * from samp_poly

In [0]:
%sql 
-- NOTICE: difference between `st_geoglength` [meters] and `st_length` [units: e.g. degrees]
select st_geoglength(g) as dbx_length_meters, st_length(g) as dbx_length_units, * from samp_line

## [3] More on Functions

__`ST_Centroid`__

> Expression takes as input a geometry object and returns the centroid of that geometry as a 2D point.

A few details about the behavior:

* If the input geometry is empty, the 2D empty point is returned.
* If the input geometry consists of points only, the centroid is the average of the X and Y coordinates of the points.
* If the input geometry contains linear segments (but no areal geometries), the centroid is the weighted average of the midpoints of the linear segments, where the weights are * the lengths of the segments.
* If the input geometry contains polygons, the centroid is the weighted average of the centroids of the polygons, where the weights are the areas of the polygons.
* In case of mixed topological dimension components, the centroid computation is based on the components of highest topological dimension.
* If the input is a BINARY value, it is expected to be the WKB description of a geometry.
* If the input is a STRING value, it is expected to be the GeoJSON or WKT description of a geometry.
* If the input is GeoJSON, the SRID of the resulting geometry is 4326.
* If the input is WKB or WKT, the SRID of the resulting geometry is 0.
* If the input is a GEOMETRY value, the SRID value of the output geometry is the same as that of the input value.

In [0]:
%sql select st_astext(st_centroid(g)) as c, g from samp_line

__`ST_Distance` | `ST_DistanceSphere` | `ST_DistanceSpheroid`__

__2D Euclidean__

<p/>

* `ST_Distance`: takes as input two geometries, or two standard representations of geometries (GeoJSON, WKB, or WKB), and returns the 2D Euclidean distance of the two geometries. The units of the returned distance are those of the input geometries. 

For this expression:
  1. If at least one of the geometries is empty, NULL is returned.
  1. If the input geospatial representations are not valid, a parse error is returned.
  1. If the two inputs are GEOMETRY values with different SRIDs, an error is returned.

__WGS84 Ellipsoid__

<p/>

* `ST_DistanceSphere`: takes as input two point geometries and returns their spherical distance in meters, measured on a sphere whose radius is the _mean radius of the WGS84 ellipsoid_. The expression can also take STRING/BINARY values as input, representing geometries in GeoJSON, WKB, or WKT format.
* `ST_DistanceSpheroid`: takes as input two point geometries and returns their geodesic distance in meters, _measured on the WGS84 ellipsoid_. The expression can also take STRING/BINARY values as input, representing geometries in GeoJSON, WKB, or WKT format.

For both of these expressions:
  1. The geometries are expected to have coordinates in degrees.
  1. The expression returns NULL is at least one of the two input point geometries is empty.
  1. The expression returns an error if at least one of the two input geometries is not a point.
  1. In the case both inputs are GEOMETRY values, they are expected to have the same SRID value. Otherwise an error is returned.

In [0]:
%sql 
CREATE OR REPLACE TEMPORARY VIEW samp_pnts AS (
  select 'POINT(-74.0060 40.7128)' as nyc, 'POINT(-77.0257 38.9005)' as dc
);

-- notice ~3.5 degrees vs 327K meters 
-- sphere uses the mean radius of the WGS84 ellipsoid
-- spheroid uses the full WGS84 ellipsoid datum
select 
  format_number(st_distance(nyc,dc), 8) as dist_degrees, 
  format_number(st_distancesphere(nyc,dc), 4) as dist_sphere_meters, 
  format_number(st_distancespheroid(nyc,dc), 4) as dist_spheroid_meters 
from samp_pnts

In [0]:
## -- uncomment to render
# map_render(spark.table("samp_pnts"), "nyc")
# map_render(spark.table("samp_pnts"), "dc")

__Convext Hull | Scale | Rotate | Translate__

> Below is a screenshot of these examples.

In [0]:
displayHTML("""<img src="..." width="100%"/>""")

In [0]:
%sql
-- convex hull
CREATE OR REPLACE TEMPORARY VIEW cvh_poly AS (
  SELECT *, st_astext(st_convexhull(g)) as cg from samp_line
);
SELECT * from cvh_poly;

In [0]:
## -- uncomment to render
# map_render(spark.table("cvh_poly"), "cg") # <- hint: adjust layer settings in kepler to see the original 'g' as well

In [0]:
%sql
-- ST_Scale: takes as input a geometry and scale factors in the X, Y, and Z directions. 
--           The expression returns the scaled geometry using the provided scaling factors. 
--           The Z scaling factor is optional. If not specified it defaults to 1. 
--           If the geometry does not have any Z coordinates, the Z scaling factor is ignored.
-- Here we scale X and Y each by 1.00003 degrees (assuming 4326).
CREATE OR REPLACE TEMPORARY VIEW scale_line AS (
  SELECT *, st_astext(st_scale(g, 1.00003, 1.00003)) as sg from samp_line
);
SELECT * from scale_line;

In [0]:
## -- uncomment to render
# map_render(spark.table("scale_line"), "sg")

In [0]:
%sql
-- ST_Rotate: takes as input a geometry and a rotation angle (in radians) and 
--            rotates the input geometry by the input angle around the Z axis.
-- Here we rotate by only 0.0000005 radians 
CREATE OR REPLACE TEMPORARY VIEW rotate_poly AS (
  SELECT *, st_astext(st_rotate(g, 0.0000005)) as rg from samp_poly
);
SELECT * from rotate_poly;

In [0]:
## -- uncomment to render
# map_render(spark.table("rotate_poly"), "rg")

In [0]:
%sql
-- ST_Translate: takes as input a geometry and offsets in the X, Y, and Z directions. 
--                The expression returns the translated geometry using the provided 
--                offsets. The Z offset is optional. If not specified it defaults to 0. 
--                If the geometry does not have any Z coordinates, the Z offset is ignored.
-- Here we translate X (longitude) by 0.00005 degrees (assuming 4326)
CREATE OR REPLACE TEMPORARY VIEW translate_poly AS (
  SELECT *, st_astext(st_translate(g, 0.00005, 0.0)) as tg from samp_poly
);
SELECT * from translate_poly;

In [0]:
## -- uncomment to render
# map_render(spark.table("translate_poly"), "tg")

__Adjust Precision + Endianness__

In [0]:
%sql 
-- you can reduce precision and format numbers
-- e.g. for various measures performed
with len_meters as (
  select st_geoglength(g) as geog_len, * from samp_line
)
select geog_len, cast(geog_len as decimal(12,6)) as less_precise_len, format_number(geog_len,2) as pretty_len from len_meters

In [0]:
%sql 
-- you can reduce precision on the spatial data itself
-- similar calls for st_astext (aka st_aswkt) and st_asgeojson
select st_astext(st_geomfromwkt(g), 4) as g_precision_4, g, * except (g) from samp_line

In [0]:
%sh 
# can verify the OS Endianness
lscpu | grep Endian

In [0]:
%sql 
-- st_asbinary (aka st_aswkb) and st_asewkb hav an optional string param 
-- to specificy endianness [little-endian ('NDR') or big-endian ('XDR')]
-- Here we specify big-endian (just as an example)
select 
    st_aswkb(st_geomfromwkt(g), 'XDR') as g,
    * except(g) 
  from samp_line

__Work with other supported languages__

> While only SQL bindings are available in v1 of the preview, you can easily incorporate with other DBR supported languages, such as Python as shown below. __Note: a very straight-forward pattern to also consider is `df = sql("""<some sql here>""")`.__

In [0]:
## -- here is an example of using python with sql bindings
##    This is adapted from logic in the hidden cell for
##    `map_render()` function defined at the top of the notebook
display(
  spark.table("samp_line")
      # - xy min/max
      .select( 
        F.expr(f"st_xmin(g) as xmin"), 
        F.expr(f"st_ymin(g) as ymin"),
        F.expr(f"st_xmax(g) as xmax"),
        F.expr(f"st_ymax(g) as ymax")
      )
      .groupBy()
        .agg(
          F.min("xmin").alias("xmin"),
          F.min("ymin").alias("ymin"),
          F.max("xmax").alias("xmax"),
          F.max("ymax").alias("ymax")
        )
      # - centroid xy ranges
      .withColumn("centroid_x", F.expr("(xmin + xmax) / 2.0"))
      .withColumn("centroid_y", F.expr("(ymin + ymax) / 2.0"))  
      .withColumn("pnt_sw", F.expr("st_astext(st_point(xmin,ymin))"))
      .withColumn("pnt_nw", F.expr("st_astext(st_point(xmin,ymax))"))
      .withColumn("pnt_se", F.expr("st_astext(st_point(xmax,ymin))"))
      .withColumn("pnt_ne", F.expr("st_astext(st_point(xmax,ymax))"))
      .withColumn(
        "width_meters", 
        F.expr("st_geoglength(st_astext(st_makeline(array( st_geomfromtext(pnt_sw), st_geomfromtext(pnt_se) ))))")
      )
      .withColumn(
        "height_meters", 
        F.expr("st_geoglength(st_astext(st_makeline(array( st_geomfromtext(pnt_sw), st_geomfromtext(pnt_nw) ))))")
      )
)

__Here is a little bit on v1 support for SRID__

> [SRID](https://en.wikipedia.org/wiki/Spatial_reference_system) (Spatial Reference System Identifier) is supported in the preview; however, we should point out the following:

<p/>

1. GEOGRAPHY data type only supports SRID=4326
1. H3 assumes SRID=4326 (WGS84 ellipsoid)
1. `ST_Transform` is the main function to reproject from one SRID to another.

_For best interoperability among various spatial data, we recommend standardizing to SRID=4326, then only specially transforming as needed for various analysis or generated data products._ __For v1 of the preview, except for `ST_Transform` no other function utilizes the information provided by the SRID.__

In [0]:
%sql 
-- when srid is not set
select st_srid(st_geomfromtext(g)) as srid, * from samp_poly

In [0]:
%sql 
-- when srid is set / inferred
-- Note: GeoJSON spec requires SRID=4326 [WGS84]
with samp_geojson as (
  select * except (g), st_asgeojson(st_geomfromtext(g)) as g from samp_poly
) 
select st_srid(st_geogfromgeojson(g)) as srid, * from samp_geojson

In [0]:
%sql 
-- let's set srid to 4326
-- we will first coerce wkt to ewkb and specify SRID
with samp_ewkb as (
  select * except (g), st_asewkb(st_geomfromtext(g, 4326)) as g from samp_poly
) 
select st_srid(st_geomfromewkb(g)) as srid, * from samp_ewkb

In [0]:
%sql 
-- lets transform from 4326 [WGS84] to 3857 [Web Mercator Projection]
-- using the same ewkb from above
with samp_ewkb as (
  select * except (g), st_asewkb(st_geomfromtext(g, 4326)) as g from samp_poly
) 
select st_astext(st_transform(st_geomfromewkb(g), 3857)) as g_3857, st_astext(st_geomfromewkb(g)) as g, * except(g) from samp_ewkb

## [4] Spatial Joins using `H3_` + `ST_` Functions

> v1 of the preview introduces (new) `h3_tessellateaswkb` which allows vector chipping of spatial data to establish a spatial index, which is essential to achieve scaled performance. _Without H3 indexing, you as a user will have a poor experience even before hitting large scales, see discussion from [previous blog](https://www.databricks.com/blog/2022/12/13/spatial-analytics-any-scale-h3-and-photon.html) on the impact of not having any index._ __v1 of the preview has no implicit / managed indexing of your spatial data.__ Read on below to understand more about how to data engineer your base geometry tables to establish spatial indexing. 

__The three screenshots below show the main progression of tessellate -> join -> analyze -> aggregate; the outputs can be rendered live if you have enabled kepler.__

In [0]:
displayHTML("""<img src="..." width="100%"/>""")

In [0]:
%sql 
-- verify h3 functions
show functions like 'h3_*'

### Setup Catalog + Schema

> We are going to persist a couple of tables to demonstrate the main hybrid tessellation pattern.

In [0]:
%sql 
-- !!! pick your own catalog + schema !!!
--     optionally: don't run this and use defaults
use catalog geospatial_docs;
use schema spatial_preview;

### Initial Trips Data [Points]

> More in [docs](https://docs.databricks.com/en/dbfs/databricks-datasets.html#browse-databricks-datasets) on databricks public datasets.

In [0]:
# - we want a row_id
# - also, stripping some bad data
# - this might take a few minutes (based on cluster size)
df_trip_samp = (
  spark.table("delta.`/databricks-datasets/nyctaxi/tables/nyctaxi_yellow`")
  .sample(fraction=0.01)
  .dropna(subset=[
    'pickup_datetime', 'dropoff_datetime',
    'pickup_longitude', 'pickup_latitude', 
    'dropoff_longitude', 'dropoff_latitude'
  ])
  .filter(F.expr("cast(pickup_latitude as int) <> 0 and cast(pickup_longitude as int) <> 0"))
  .selectExpr("""xxhash64(
    pickup_datetime, dropoff_datetime,
    pickup_longitude, pickup_latitude, 
    dropoff_longitude, dropoff_latitude)
    as trip_row_id""",
    "*"
  ) 
)

df_trip_samp.createOrReplaceTempView("trip")

In [0]:
%sql 
-- working with ~12.2M trips
select format_number(count(1),0) as trip_cnt from trip

In [0]:
%sql select * from trip limit 10

__Add additional h3 `cellid` column to trips data__

> We are going with h3 resolution `6` (without much explanation) through existing function [h3_longlatash3](https://docs.databricks.com/en/sql/language-manual/functions/h3_longlatash3.html) (suitable for points); for serious use, you will want to reason about best resolution(s) for your data, see [h3 res statistics](https://h3geo.org/docs/core-library/restable/) for more. Also, notice the use of `CLUSTER BY` on table creation, followed by `OPTIMIZE` to easily apply [Liquid Clustering](https://docs.databricks.com/en/delta/clustering.html) to 'pickup_cellid'.

In [0]:
%sql
-- table will use liquid clustering
CREATE OR REPLACE TABLE trip_h3 CLUSTER BY (pickup_cellid) AS 
(
  SELECT h3_longlatash3(pickup_longitude, pickup_latitude, 6) AS pickup_cellid, * 
  FROM trip
);

-- execute liquid clustering
OPTIMIZE trip_h3;

SELECT * FROM trip_h3 LIMIT 10;

__Look closer at trips data__

> The average area for resolution 6 is `36KM^2`, so with a quick plot of NYC, we can see that there are multiple outliers above ~1M which we would want to remain mindful of when considering join skew (especially since this is only a sample). __For now, we move ahead!__

In [0]:
%sql select pickup_cellid, count(1) as count from trip_h3 group by pickup_cellid order by count desc

Databricks visualization. Run in Databricks to view.

### Initial States Data [Polygons]

> Focusing on (preview function) `h3_tessellateaswkb` to setup index on non-point data. There is also [h3_coverash3](https://docs.databricks.com/en/sql/language-manual/functions/h3_coverash3.html) and [h3_polyfillash3](https://docs.databricks.com/en/sql/language-manual/functions/h3_polyfillash3.html) both having strengths for "pure" discrete / approximate use cases; for this we are going with a hybrid data engineering pattern which prepares data 1x for both precise and approximate use cases, more background [here](https://www.databricks.com/blog/2022/12/13/spatial-analytics-any-scale-h3-and-photon.html) [__don't focus on the other libraries from the blog, this is now available in the preview__]. 

__We will start with NY and NJ state polygons__

In [0]:
nj_wkt = "POLYGON((-74.6950 41.3572,-74.6559 41.3394,-73.8940 40.9934,-73.9586 40.8398,-74.0094 40.7691,-74.0231 40.6994,-74.0437 40.6786,-74.0808 40.6515,-74.1357 40.6421,-74.1962 40.6452,-74.2003 40.5952,-74.2195 40.5566,-74.2552 40.4877,-74.2264 40.4762,-73.9503 40.5253,-73.8885 40.4846,-73.9352 40.0045,-74.0410 39.6131,-74.2209 39.4744,-74.6713 38.9882,-74.8553 38.8664,-75.0476 38.8472,-75.1685 39.0565,-75.3250 39.2525,-75.5544 39.4500,-75.5612 39.4966,-75.5283 39.4998,-75.5338 39.5411,-75.5090 39.5761,-75.5708 39.6237,-75.5104 39.6713,-75.4843 39.7167,-75.4156 39.8033,-75.2632 39.8360,-75.1918 39.8823,-74.7922 40.1180,-74.7331 40.1390,-74.8485 40.2565,-74.9419 40.3361,-74.9721 40.4020,-75.0627 40.4240,-75.0613 40.4898,-75.1067 40.5733,-75.2138 40.5639,-75.2028 40.6192,-75.2069 40.6494,-75.0806 40.8284,-75.0998 40.8429,-75.0504 40.8689,-75.1369 40.9913,-74.8677 41.2293,-74.7537 41.3479,-74.7249 41.3469,-74.6960 41.3593,-74.6950 41.3572))"

ny_wkt = "MULTIPOLYGON (((-79.7621 42.5146, -79.7621 42.5143, -79.7624 42.5142, -79.7621 42.5146)), ((-78.9313 42.8508, -78.9024 42.9061, -78.9313 42.9554, -78.9656 42.9584, -79.0219 42.9886, -79.0027 43.0568, -79.0727 43.0769, -79.0713 43.1220, -79.0302 43.1441, -79.0576 43.1801, -79.0604 43.2482, -79.0837 43.2812, -79.2004 43.4509, -78.6909 43.6311, -76.7958 43.6321, -76.4978 43.9987, -76.4388 44.0965, -76.3536 44.1349, -76.3124 44.1989, -76.2437 44.2049, -76.1655 44.2413, -76.1353 44.2973, -76.0474 44.3327, -75.9856 44.3553, -75.9196 44.3749, -75.8730 44.3994, -75.8221 44.4308, -75.8098 44.4740, -75.7288 44.5425, -75.5585 44.6647, -75.4088 44.7672, -75.3442 44.8101, -75.3058 44.8383, -75.2399 44.8676, -75.1204 44.9211, -74.9995 44.9609, -74.9899 44.9803, -74.9103 44.9852, -74.8856 45.0017, -74.8306 45.0153, -74.7633 45.0046, -74.7070 45.0027, -74.5642 45.0007, -74.1467 44.9920, -73.7306 45.0037, -73.4203 45.0085, -73.3430 45.0109, -73.3547 44.9874, -73.3379 44.9648, -73.3396 44.9160, -73.3739 44.8354, -73.3324 44.8013, -73.3667 44.7419, -73.3873 44.6139, -73.3736 44.5787, -73.3049 44.4916, -73.2953 44.4289, -73.3365 44.3513, -73.3118 44.2757, -73.3818 44.1980, -73.4079 44.1142, -73.4367 44.0511, -73.4065 44.0165, -73.4079 43.9375, -73.3749 43.8771, -73.3914 43.8167, -73.3557 43.7790, -73.4244 43.6460, -73.4340 43.5893, -73.3969 43.5655, -73.3818 43.6112, -73.3049 43.6271, -73.3063 43.5764, -73.2582 43.5675, -73.2445 43.5227, -73.2582 43.2582, -73.2733 42.9715, -73.2898 42.8004, -73.2664 42.7460, -73.3708 42.4630, -73.5095 42.0840, -73.4903 42.0218, -73.4999 41.8808, -73.5535 41.2953, -73.4834 41.2128, -73.7275 41.1011, -73.6644 41.0237, -73.6578 40.9851, -73.6132 40.9509, -72.4823 41.1869, -72.0950 41.2551, -71.9714 41.3005, -71.9193 41.3108, -71.7915 41.1838, -71.7929 41.1249, -71.7517 41.0462, -72.9465 40.6306, -73.4628 40.5368, -73.8885 40.4887, -73.9490 40.5232, -74.2271 40.4772, -74.2532 40.4861, -74.1866 40.6468, -74.0547 40.6556, -74.0156 40.7618, -73.9421 40.8699, -73.8934 40.9980, -73.9854 41.0343, -74.6274 41.3268, -74.7084 41.3583, -74.7101 41.3811, -74.8265 41.4386, -74.9913 41.5075, -75.0668 41.6000, -75.0366 41.6719, -75.0545 41.7672, -75.1945 41.8808, -75.3552 42.0013, -75.4266 42.0003, -77.0306 42.0013, -79.7250 41.9993, -79.7621 42.0003, -79.7621 42.1827, -79.7621 42.5143, -79.0672 42.7783, -78.9313 42.8508)))"

state_df = spark.createDataFrame(
  [(1, "New Jersey", nj_wkt),(2, "New York", ny_wkt)], 
  ["state_row_id", "state", "g"]
)
state_df.createOrReplaceTempView("state_poly")

In [0]:
%sql 
-- quick check to make sure spatial data is valid
select *, st_isvalid(g) as test_valid from state_poly

In [0]:
## -- uncomment to render
# map_render(state_df, "g")

__H3 Index on States__

> First we will show some variations on executing `h3_tessellateaswkb`; then, we will actually generate a table which should make it clear as to why #3 option is recommended.

In [0]:
%sql 
-- THREE OPTIONS [#3 RECOMMENDED]
-- This is just a "dry-run" for you to see / understand the options
-- Hint: change commenting to view other options [#3 PRE-SELECTED]

-- [1] tessellating at h3 resolution 6
--     returns an array per geom, so doesn't explode or perform table shredding
--     !!! This is not recommended for scaled performance !!!
-- select h3_tessellateaswkb(g, 6) as tessellate, * from state_poly

-- [2] tessellating at h3 resolution 6 + `explode`
--     returns a row per h3 cell per geom, but doesn't perform struct shredding
--     !!! This is not recommended for optimized table ordering, e.g. with liquid clustering !!!
-- select explode(h3_tessellateaswkb(g, 6)) as tessellate, * except (g) from state_poly

-- [3] tessellating at h3 resolution 6 + `inline`
--     returns a row per h3 cell per geom with struct shredding, meaning:
--     'cellid', 'core', and 'chip' become top-level fields
--     !!! This is recommended for scaled performance and optimized table ordering !!!
select inline(h3_tessellateaswkb(g, 6)), * except (g) from state_poly where state = "New York" limit 10

In [0]:
%sql
-- table will use liquid clustering
-- The use of inline allows us to cluster by 'cellid'
-- NOTICE: we do not include the full geometry 'g' here;
--         rather, keep only vector chips per cellid
CREATE OR REPLACE TABLE state_h3 CLUSTER BY (cellid) AS 
(
  SELECT state_row_id, INLINE(h3_tessellateaswkb(g, 6)), * EXCEPT(state_row_id, g) 
  FROM state_poly
);

-- execute liquid clustering
OPTIMIZE state_h3;

SELECT * FROM state_h3 LIMIT 10;

In [0]:
%sql 
-- we see that the 2 states are now converted into 4.6K vector chips
select format_number(count(1),0) as tessellate_cnt from state_h3

_Let's render both the states and their h3 cells_

In [0]:
## -- uncomment to render
# map_render_dfMapItems(
#   DFMapItem(spark.table("state_h3"), "cellid", RENDER_TYPE.H3_INT, exclude_cols=["chip"]), 
#   DFMapItem(spark.table("state_poly"), "g", RENDER_TYPE.GEOMETRY, geo_format=GEO_FORMAT.WKT)
# )

### Spatial Join: Point-in-Polygon

> Now that our base vector tables have been spatially indexed (tables `trip_h3` and `state_h3`), .

In [0]:
%sql 
CREATE OR REPLACE TABLE pickup_state_h3 CLUSTER BY (cellid) AS (
SELECT
  state_h3.*,
  t.*
from
  (select /*+ SKEW('pickup_cellid') */ * from trip_h3) as t
  join state_h3 
  on cellid == pickup_cellid
  where 
    core or 
    st_contains(st_geomfromwkb(chip), st_point(pickup_longitude, pickup_latitude))
);

OPTIMIZE pickup_state_h3;
 
select format_number(count(1),0) as count from pickup_state_h3;

In [0]:
%sql select * from pickup_state_h3 limit 10;

__How many pickups were in New Jersey vs New York?__

> We find that 12M+ were in NY with ~20K in NJ.

In [0]:
%sql
select
  state,
  count(1) as pickup_cnt
from
  pickup_state_h3
group by
  state
order by
  state;

Databricks visualization. Run in Databricks to view.

### H3: FILTER + UNION Results

> We can strip out the H3 specifics and/or union back the results.

In [0]:
%sql 
-- we can easily filter out the h3 related fields to view the spatial join results
select * except(cellid, core, chip, pickup_cellid) from pickup_state_h3 limit 10

In [0]:
%sql 
-- we can also union h3 chips back together, e.g. ones involved in the results
-- note: this will not be the entire state polygon due to the nature of
-- performing an INNER join in pickup_state_h3
CREATE OR REPLACE TEMPORARY VIEW state_chip_union AS (
  select state_row_id, state, st_astext(st_union_agg(chip)) as chip_union from 
  (select distinct state_row_id, state, chip from pickup_state_h3) 
  group by state_row_id, state
  order by state_row_id
);

select * from state_chip_union;

In [0]:
## -- uncomment to render
# map_render(spark.table("state_chip_union"), "chip_union")

_The ^ is just a union of h3 cells per state where there was any pickup activity._ 

__Notes:__ 

> Option-1: if you wanted to rebuild the original geospatial feature from the h3 vector chips, you can change from 'pickup_state_h3' being produced by an `INNER` join to a `RIGHT OUTER` join (results increase by ~4k as all the h3 WKB chips without any pickups are preserved).

```
-- change to RIGHT OUTER join (from INNER)
-- to preserve polygon chips
CREATE OR REPLACE TEMPORARY VIEW pickup_state_h3_outer AS (
SELECT
  state_h3.*,
  t.*
from
  (select /*+ SKEW('pickup_cellid') */ * from trip_h3) as t
  right outer join state_h3 
  on cellid == pickup_cellid
  where 
    core or 
    st_contains(st_geomfromwkb(chip), st_point(pickup_longitude, pickup_latitude))
)
```

> Option-2 [shown below]: you can always join back the original geometry; __however, this makes most sense when performing aggregates.__

In [0]:
%sql 
-- join back the original geometry
-- most suitable when performing aggregates
CREATE OR REPLACE TEMPORARY VIEW state_agg AS (
  SELECT
    t.*,
    state_poly.g
  FROM (
    SELECT 
      state_row_id, state, count(1) as pickup_cnt, 
      format_number(count(1),0) as display_cnt 
      FROM pickup_state_h3
    GROUP BY state_row_id, state
  ) as t
  join state_poly
  on t.state_row_id == state_poly.state_row_id
 
  ORDER BY state_row_id
);

select * from state_agg;

In [0]:
## -- uncomment to render
# map_render(spark.table("state_agg"), "g", exclude_cols=['pickup_cnt'])